# Chronos-2 walk-forward PCA with native REINFORCE

This notebook shows the leakage-safe Chronos plus PCA workflow supported by `crosslearn`:

1. build aligned Chronos embeddings with `embed_dataframe(...)`
2. reduce those embeddings with `walkforward_pca_dataframe(...)`
3. train offline on the resulting `pca_*` columns
4. evaluate the same representation online with `WalkForwardChronosPCAWrapper`


In [ ]:
# Uncomment in a fresh environment:
# %pip install "crosslearn[chronos]" gym-anytrading pandas matplotlib

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from gym_anytrading.datasets import STOCKS_GOOGL
from gym_anytrading.envs import StocksEnv

from crosslearn import REINFORCE, make_vec_env
from crosslearn.envs import WalkForwardChronosPCAWrapper
from crosslearn.extractors import (
    WalkForwardPCATransformer,
    embed_dataframe,
    walkforward_pca_dataframe,
)

In [ ]:
LOOKBACK = 30
PCA_WARMUP = 900
FEATURE_COLUMNS = ['Open', 'High', 'Low', 'Close', 'Volume']
SELECTED_COLUMNS = ['Close', 'Volume']
PCA_FRAME_BOUND = (LOOKBACK, len(STOCKS_GOOGL))
ONLINE_FRAME_BOUND = (LOOKBACK + PCA_WARMUP, len(STOCKS_GOOGL))

print('PCA frame bound:', PCA_FRAME_BOUND)
print('Agent-visible online frame bound:', ONLINE_FRAME_BOUND)

## Offline Chronos embeddings

The first stage keeps Chronos separate from PCA. `embed_dataframe(...)` appends aligned `chronos_*` columns to the trimmed dataframe.


In [ ]:
embedded_df = embed_dataframe(
    STOCKS_GOOGL,
    lookback=LOOKBACK,
    frame_bound=PCA_FRAME_BOUND,
    feature_columns=FEATURE_COLUMNS,
    selected_columns=SELECTED_COLUMNS,
    progress_bar=True,
)

display(embedded_df.head())
print('Chronos columns:', embedded_df.filter(like='chronos_').shape[1])
print('Rows after embedding:', len(embedded_df))

## Walk-forward PCA on Chronos embeddings

The PCA stage is leakage-safe. At each row, the mean, optional standard deviation, and PCA loadings are fit only on earlier Chronos rows.


In [ ]:
transformer_preview = WalkForwardPCATransformer(
    warmup=PCA_WARMUP,
    explained_variance_threshold=0.99,
    standardize=True,
    solver='svd',
    expanding_warmup=True,
    compute_dtype=torch.float64,
    device='auto',
    batch_size=64,
)
transformer_preview.fit(
    embedded_df.filter(like='chronos_').to_numpy(dtype=np.float32)
)

initial_ratio = transformer_preview.initial_explained_variance_ratio_
cumulative_ratio = np.cumsum(initial_ratio, dtype=np.float64)
plt.figure(figsize=(10, 4))
plt.plot(cumulative_ratio)
plt.axhline(0.99, color='tab:red', linestyle='--', label='99% threshold')
plt.axvline(transformer_preview.n_components_ - 1, color='tab:green', linestyle='--', label='chosen n_components')
plt.title('Initial warmup cumulative explained variance')
plt.xlabel('component index')
plt.ylabel('cumulative explained variance')
plt.legend()
plt.show()
print('Chosen fixed n_components:', transformer_preview.n_components_)

In [ ]:
pca_df = walkforward_pca_dataframe(
    embedded_df,
    feature_columns=embedded_df.filter(like='chronos_').columns.tolist(),
    warmup=PCA_WARMUP,
    explained_variance_threshold=0.99,
    standardize=True,
    solver='svd',
    expanding_warmup=True,
    compute_dtype=torch.float64,
    device='auto',
    batch_size=64,
    output_prefix='pca_',
    drop_feature_columns=True,
    trim_warmup=True,
    progress_bar=True,
)

display(pca_df.head())
print('PCA columns:', pca_df.filter(like='pca_').shape[1])
print('Rows after warmup trimming:', len(pca_df))

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(pca_df.filter(like='pca_').iloc[:, 0].to_numpy(dtype=np.float32))
plt.title('First walk-forward PCA score')
plt.xlabel('Trimmed timestep')
plt.ylabel('pca_0')
plt.show()

## Offline training on `pca_*`

Once the adaptive Chronos plus PCA features are finalized, the downstream environment sees plain numeric vectors.


In [ ]:
def render_single_episode(env, agent):
    obs, _ = env.reset()
    terminated = False
    truncated = False
    
    while not (terminated or truncated):
        action = agent.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Info: {info}")
    env.unwrapped.render_all()
    plt.show()
    env.close()

In [ ]:
class OfflineStocksEnv(StocksEnv):
    def __init__(self, prices, signal_features, **kwargs):
        self._prices = prices
        self._signal_features = signal_features.astype(np.float32)
        super().__init__(**kwargs)

    def _process_data(self):
        return self._prices, self._signal_features


def make_offline_env():
    return OfflineStocksEnv(
        prices=pca_df['Close'].to_numpy(dtype=np.float32),
        signal_features=pca_df.filter(like='pca_').to_numpy(dtype=np.float32),
        df=pca_df,
        window_size=1,
        frame_bound=(1, len(pca_df)),
    )


offline_agent = REINFORCE(make_vec_env(make_offline_env, n_envs=1), n_steps=512, seed=42, verbose=2)
offline_agent.learn(total_timesteps=10_000)

In [ ]:
render_single_episode(make_offline_env(), offline_agent)

## Online env-side Chronos + PCA

For the adaptive online path, the wrapper skips the first `LOOKBACK + PCA_WARMUP` observations by design. The next observation is the first agent-visible PCA-ready observation, so the dataset must contain at least `LOOKBACK + PCA_WARMUP + 1` rows to make the wrapper usable. After that initial skip, the wrapper keeps a walk-forward embedding history inside the environment layer, embeds one new raw window at a time, and refits PCA on the selected history window at each environment step. `solver`, `expanding_warmup`, `compute_dtype`, and `pca_device` control the PCA side independently of Chronos placement.


In [ ]:
def online_process_data(env):
    start = env.frame_bound[0] - env.window_size
    end = env.frame_bound[1]
    prices = env.df.loc[:, 'Close'].to_numpy()[start:end]
    signal_features = env.df.loc[:, FEATURE_COLUMNS].to_numpy(dtype=np.float32)[start:end]
    return prices, signal_features


class OnlineStocksEnv(StocksEnv):
    _process_data = online_process_data


def make_online_env():
    base_env = OnlineStocksEnv(
        df=STOCKS_GOOGL,
        window_size=LOOKBACK,
        frame_bound=ONLINE_FRAME_BOUND,
    )
    return WalkForwardChronosPCAWrapper(
        base_env,
        lookback=LOOKBACK,
        warmup=PCA_WARMUP,
        feature_columns=FEATURE_COLUMNS,
        selected_columns=SELECTED_COLUMNS,
        solver='svd',
        expanding_warmup=True,
        compute_dtype=torch.float64,
        device_map='auto',
        pca_device='cpu',
    )

sample_env = make_online_env()
print('Online observation space:', sample_env.observation_space)
sample_env.close()

In [ ]:
online_agent = REINFORCE(make_vec_env(make_online_env, n_envs=1), n_steps=512, seed=42, verbose=2)
online_agent.learn(total_timesteps=10_000)

In [ ]:
render_single_episode(make_online_env(), online_agent)